# Day 53: Lab - Model Tuning Mini-Project

## Project Goal
In real projects, a model you fit with default settings is rarely the best you can do — hyperparameters like tree depth, number of estimators, or class weighting can move accuracy and F1 by several points. This lab takes the churn dataset from Day 48 and works through a proper tuning workflow: build a baseline, search for better hyperparameters with cross-validation, then confirm the tuned model actually beats the baseline on the metric you care about. The point isn't just to run `GridSearchCV` — it's to see the discipline of always comparing "before" and "after" so you don't fool yourself into thinking a change helped when it didn't. Keep an eye on which metric you're optimizing for (here, F1 on the churn class) since accuracy alone can be misleading on imbalanced data.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)  # fixes randomness so the dataset is identical on every run
n = 500
df = pd.DataFrame({
    'tenure_months': np.random.randint(1, 60, n),
    'monthly_charges': np.round(np.random.uniform(20, 120, n), 2),
    'support_calls': np.random.poisson(2, n),
})
# Build churn probability with a logistic (sigmoid) function: more support calls and
# shorter tenure push probability up, longer tenure pulls it down.
churn_prob = 1 / (1 + np.exp(-(-2 + df['support_calls'] * 0.5 - df['tenure_months'] * 0.03)))
# Sample actual 0/1 churn labels from those probabilities (binomial with 1 trial = Bernoulli draw)
df['churn'] = np.random.binomial(1, churn_prob)

## Step 1: Baseline Model
Before tuning anything, you need a reference point to measure improvement against — otherwise "better" is meaningless. This cell splits the data with stratification (so the churn/non-churn ratio is preserved in both train and test sets) and fits a `RandomForestClassifier` with its default hyperparameters. The printed accuracy is the number every later change must beat; if a tuned model doesn't clear this bar, the tuning wasn't worth it. Always fit the baseline on the same train/test split you'll use later so the comparison is fair.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['churn'])
y = df['churn']
# stratify=y keeps the same churn/no-churn proportion in train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Fit with default hyperparameters — this is our "before tuning" reference score
baseline = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print('Baseline accuracy:', baseline.score(X_test, y_test))

## Step 2: Grid Search Tuning
`GridSearchCV` exhaustively tries every combination of hyperparameter values you give it, evaluating each with cross-validation, and keeps the combination that scores best — this removes the guesswork of hand-picking values. Here it searches over the number of trees (`n_estimators`), how deep each tree can grow (`max_depth`, where `None` means unlimited), and `class_weight`, which up-weights the minority class so the model doesn't just default to predicting "no churn" for everyone. Cross-validation (`cv=5`) splits the training data into 5 folds and averages performance across them, giving a more reliable estimate than a single train/validation split. We score on `f1` rather than accuracy because F1 balances precision and recall, which matters more when churners are a minority class.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Every combination of these values will be tried: 2 x 3 x 2 = 12 combinations,
# each evaluated with 5-fold cross-validation (60 model fits total)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, None],
    'class_weight': [None, 'balanced']
}
# scoring='f1' tells GridSearchCV to pick the combination with the best F1 score,
# not the best accuracy — important since churn is the minority class
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)

## Step 3: Compare Before & After
Tuning is only useful if it actually improves the metric you're optimizing for on held-out data, so this step evaluates both the baseline and the tuned model (`grid.best_estimator_`, the model refit on the full training set with the best-found parameters) on the same test set. Comparing F1 scores side by side tells you whether the extra complexity of grid search was worth it. If the tuned F1 is only marginally higher, that's a useful and common result — it means the model was already close to its ceiling given the current features, and further gains would more likely come from better features than from more tuning.

In [ ]:
from sklearn.metrics import f1_score

base_preds = baseline.predict(X_test)
# grid.best_estimator_ is the RandomForestClassifier refit with the best params found above
tuned_preds = grid.best_estimator_.predict(X_test)
print('Baseline F1:', f1_score(y_test, base_preds))
print('Tuned F1:', f1_score(y_test, tuned_preds))